# Punto 6: Consultas Analíticas con SparkSQL (AWS Glue Interactive Notebook)
Este notebook ejecuta las 5 consultas de negocio sobre la base de datos `energia_col_trusted` en AWS Glue Data Catalog usando el motor Apache Spark.

In [ ]:
%glue_version 4.0
%idle_timeout 15
%number_of_workers 2
%worker_type G.1X

In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

# Conectar con el catálogo de Glue
spark.sql("USE energia_col_trusted")
print("Base de datos seleccionada: energia_col_trusted")
spark.sql("SHOW TABLES").show()

## Pregunta 1: Concentración del Mercado Mayorista de Combustibles

In [ ]:
q1_sql = """
WITH despacho_por_departamento AS (
    SELECT
        departamento,
        producto,
        SUM(volumen_despachado) AS volumen_total_despachado,
        COUNT(*) AS total_transacciones_despacho
    FROM distribuidores_mayoristas
    WHERE CAST(anio_despacho AS INT) >= 2022
    GROUP BY departamento, producto
),
totales_nacionales AS (
    SELECT
        departamento,
        producto,
        volumen_total_despachado,
        total_transacciones_despacho,
        SUM(volumen_total_despachado) OVER (PARTITION BY producto) AS volumen_nacional_producto,
        ROUND((volumen_total_despachado / SUM(volumen_total_despachado) OVER (PARTITION BY producto)) * 100, 2) AS pct_participacion_nacional,
        DENSE_RANK() OVER (PARTITION BY producto ORDER BY volumen_total_despachado DESC) AS ranking_departamento
    FROM despacho_por_departamento
)
SELECT
    ranking_departamento,
    departamento,
    producto,
    ROUND(volumen_total_despachado, 2) AS volumen_despachado_galones,
    ROUND(volumen_nacional_producto, 2) AS volumen_nacional_galones,
    pct_participacion_nacional,
    total_transacciones_despacho
FROM totales_nacionales
WHERE ranking_departamento <= 5
ORDER BY producto, ranking_departamento
"""

df_q1 = spark.sql(q1_sql)
df_q1.show(25, truncate=False)

## Pregunta 2: Impacto y Ejecución de Subsidios en Zonas de Frontera

In [ ]:
q2_sql = """
WITH consumo_estaciones AS (
    SELECT
        codigo_municipio_dane_destino,
        departamento,
        municipio,
        anio_despacho,
        SUM(CASE WHEN zona_frontera = 'SI' THEN volumen_despachado ELSE 0.0 END) AS volumen_frontera,
        SUM(CASE WHEN zona_frontera = 'NO' THEN volumen_despachado ELSE 0.0 END) AS volumen_no_frontera,
        SUM(volumen_despachado) AS volumen_total,
        COUNT(DISTINCT CASE WHEN zona_frontera = 'SI' THEN subtipo_comprador END) AS eds_frontera_activas
    FROM estaciones_servicio
    WHERE CAST(anio_despacho AS INT) >= 2022
    GROUP BY codigo_municipio_dane_destino, departamento, municipio, anio_despacho
),
cupos_asignados AS (
    SELECT
        codigo_municipio_dane_destino,
        anio_beneficio,
        SUM(beneficio) AS cupo_total_upme_asignado,
        COUNT(*) AS total_asignaciones_cupo
    FROM beneficios_frontera
    WHERE CAST(anio_beneficio AS INT) >= 2022
    GROUP BY codigo_municipio_dane_destino, anio_beneficio
)
SELECT
    c.anio_despacho AS anio,
    c.departamento,
    c.municipio,
    c.codigo_municipio_dane_destino,
    ROUND(c.volumen_frontera, 2) AS volumen_despachado_frontera,
    ROUND(COALESCE(b.cupo_total_upme_asignado, 0.0), 2) AS cupo_upme_asignado,
    CASE 
        WHEN COALESCE(b.cupo_total_upme_asignado, 0.0) > 0 
        THEN ROUND((c.volumen_frontera / b.cupo_total_upme_asignado) * 100, 2)
        ELSE NULL 
    END AS pct_ejecucion_cupo,
    ROUND(c.volumen_no_frontera, 2) AS volumen_despachado_ordinario,
    c.eds_frontera_activas
FROM consumo_estaciones c
LEFT JOIN cupos_asignados b 
    ON c.codigo_municipio_dane_destino = b.codigo_municipio_dane_destino 
   AND c.anio_despacho = b.anio_beneficio
WHERE c.volumen_frontera > 0
ORDER BY pct_ejecucion_cupo DESC
LIMIT 20
"""

df_q2 = spark.sql(q2_sql)
df_q2.show(20, truncate=False)

## Pregunta 3: Correlación entre Infraestructura y Ventas de GNCV

In [ ]:
q3_sql = """
WITH agentes_municipio AS (
    SELECT
        codigo_municipio_dane,
        departamento,
        municipio,
        COUNT(*) AS total_agentes_autorizados,
        COUNT(CASE WHEN tipo_agente LIKE '%TALLER%' THEN 1 END) AS talleres_conversion,
        COUNT(CASE WHEN tipo_agente LIKE '%ESTACION%' THEN 1 END) AS estaciones_gncv
    FROM agentes_gncv
    WHERE estado_agente = 'ACTIVO'
    GROUP BY codigo_municipio_dane, departamento, municipio
),
ventas_municipio AS (
    SELECT
        codigo_municipio_dane,
        SUM(cantidad_volumen_suministrado) AS volumen_total_m3,
        SUM(numero_de_ventas) AS transacciones_totales,
        SUM(vehiculos_atendidos) AS vehiculos_totales,
        AVG(eds_activas) AS promedio_eds_operativas
    FROM ventas_gncv
    GROUP BY codigo_municipio_dane
)
SELECT
    a.departamento,
    a.municipio,
    a.codigo_municipio_dane,
    a.total_agentes_autorizados,
    a.talleres_conversion,
    a.estaciones_gncv,
    ROUND(COALESCE(v.volumen_total_m3, 0.0), 2) AS volumen_gncv_suministrado_m3,
    COALESCE(v.vehiculos_totales, 0) AS total_vehiculos_atendidos,
    CASE 
        WHEN a.total_agentes_autorizados > 0 
        THEN ROUND(COALESCE(v.volumen_total_m3, 0.0) / a.total_agentes_autorizados, 2)
        ELSE 0.0 
    END AS volumen_promedio_por_agente_m3
FROM agentes_municipio a
LEFT JOIN ventas_municipio v 
    ON a.codigo_municipio_dane = v.codigo_municipio_dane
ORDER BY volumen_gncv_suministrado_m3 DESC
LIMIT 25
"""

df_q3 = spark.sql(q3_sql)
df_q3.show(25, truncate=False)

## Pregunta 4: Trazabilidad y Cumplimiento de Biocombustibles (B100 / Etanol)

In [ ]:
q4_sql = """
WITH produccion_biocombustibles AS (
    SELECT
        anio_despacho,
        mes_despacho,
        SUM(CASE WHEN producto LIKE '%BIODIESEL%' OR producto LIKE '%B100%' THEN volumen_despachado ELSE 0.0 END) AS produccion_b100_galones,
        SUM(CASE WHEN producto LIKE '%ETANOL%' OR producto LIKE '%ALCOHOL%' THEN volumen_despachado ELSE 0.0 END) AS produccion_etanol_galones,
        SUM(volumen_despachado) AS produccion_total_bio_galones
    FROM productores_b100e
    GROUP BY anio_despacho, mes_despacho
),
demanda_despacho_mayorista AS (
    SELECT
        anio_despacho,
        mes_despacho,
        SUM(CASE WHEN producto LIKE '%BIODIESEL%' THEN volumen_despachado ELSE 0.0 END) AS despacho_diesel_mezcla,
        SUM(CASE WHEN producto LIKE '%GASOLINA%' OR producto LIKE '%CORRIENTE%' THEN volumen_despachado ELSE 0.0 END) AS despacho_gasolina_mezcla,
        SUM(volumen_despachado) AS despacho_total_combustibles
    FROM distribuidores_mayoristas
    GROUP BY anio_despacho, mes_despacho
)
SELECT
    d.anio_despacho,
    d.mes_despacho,
    ROUND(d.despacho_diesel_mezcla * 0.10, 2) AS demanda_teorica_b100_10pct,
    ROUND(COALESCE(p.produccion_b100_galones, 0.0), 2) AS produccion_real_b100,
    CASE 
        WHEN (d.despacho_diesel_mezcla * 0.10) > 0 
        THEN ROUND((COALESCE(p.produccion_b100_galones, 0.0) / (d.despacho_diesel_mezcla * 0.10)) * 100, 2)
        ELSE NULL 
    END AS pct_cobertura_b100,
    ROUND(d.despacho_gasolina_mezcla * 0.10, 2) AS demanda_teorica_etanol_10pct,
    ROUND(COALESCE(p.produccion_etanol_galones, 0.0), 2) AS produccion_real_etanol,
    CASE 
        WHEN (d.despacho_gasolina_mezcla * 0.10) > 0 
        THEN ROUND((COALESCE(p.produccion_etanol_galones, 0.0) / (d.despacho_gasolina_mezcla * 0.10)) * 100, 2)
        ELSE NULL 
    END AS pct_cobertura_etanol
FROM demanda_despacho_mayorista d
LEFT JOIN produccion_biocombustibles p 
    ON d.anio_despacho = p.anio_despacho 
   AND d.mes_despacho = p.mes_despacho
WHERE CAST(d.anio_despacho AS INT) >= 2022
ORDER BY d.anio_despacho DESC, d.mes_despacho DESC
LIMIT 25
"""

df_q4 = spark.sql(q4_sql)
df_q4.show(25, truncate=False)

## Pregunta 5: Macro-tendencias Energéticas Cruzadas (Electricidad vs Gas vs Combustibles)

In [ ]:
q5_sql = """
WITH generacion_electrica AS (
    SELECT
        year,
        month,
        SUM(CASE WHEN product = 'Hydro' THEN value ELSE 0.0 END) AS gen_hidro_gwh,
        SUM(CASE WHEN product IN ('Solar', 'Wind', 'Other renewables') THEN value ELSE 0.0 END) AS gen_renovable_no_convencional_gwh,
        SUM(CASE WHEN product IN ('Coal', 'Natural gas', 'Oil') THEN value ELSE 0.0 END) AS gen_termica_fosil_gwh,
        SUM(value) AS gen_total_gwh
    FROM iea_electricity
    WHERE CAST(year AS INT) BETWEEN 2018 AND 2022
    GROUP BY year, month
),
consumo_gas_mensual AS (
    SELECT
        anio_venta AS year,
        mes_venta AS month,
        SUM(cantidad_volumen_suministrado) AS gncv_suministrado_m3,
        SUM(numero_de_ventas) AS transacciones_gncv
    FROM ventas_gncv
    GROUP BY anio_venta, mes_venta
),
despacho_combustibles_mensual AS (
    SELECT
        anio_despacho AS year,
        mes_despacho AS month,
        SUM(volumen_despachado) AS combustible_liquido_galones
    FROM distribuidores_mayoristas
    GROUP BY anio_despacho, mes_despacho
)
SELECT
    e.year,
    e.month,
    ROUND(e.gen_hidro_gwh, 2) AS generacion_hidroelectrica_gwh,
    ROUND(e.gen_termica_fosil_gwh, 2) AS generacion_termica_gwh,
    ROUND(e.gen_total_gwh, 2) AS generacion_total_gwh,
    ROUND(COALESCE(g.gncv_suministrado_m3, 0.0), 2) AS demanda_gncv_m3,
    ROUND(COALESCE(c.combustible_liquido_galones, 0.0), 2) AS despacho_combustibles_galones
FROM generacion_electrica e
LEFT JOIN consumo_gas_mensual g 
    ON CAST(e.year AS INT) = CAST(g.year AS INT)
   AND CAST(e.month AS INT) = CAST(g.month AS INT)
LEFT JOIN despacho_combustibles_mensual c 
    ON CAST(e.year AS INT) = CAST(c.year AS INT)
   AND CAST(e.month AS INT) = CAST(c.month AS INT)
ORDER BY e.year DESC, e.month DESC
LIMIT 30
"""

df_q5 = spark.sql(q5_sql)
df_q5.show(30, truncate=False)